# 🎓 Day 1 · Session 2
# Working with Modern LLMs
## Frontier Models, Open-Source Models & Universal APIs

In Session 1, we understood how LLMs work internally.  
In this session, we move from **understanding models** to **using models through code**.

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- Understand cloud, local and hybrid LLM usage
- Call OpenAI models using Python
- Understand `system`, `user`, and `assistant` message roles
- Compare model behavior using the same prompt
- Use streaming responses
- Build a simple multi-turn teaching assistant
- Run an open-source model locally using Ollama
- Apply a model selection framework

## 📦 Install Required Packages

Run only if packages are missing.

```bash
pip install openai python-dotenv pandas
```

In [14]:
# Uncomment if required
# %pip install openai python-dotenv pandas
#Ollama is installed separately on the Mac and runs locally at:
#http://localhost:11434

## 📚 Imports

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
from openai import OpenAI

## 🔑 Load API Key from `.env`

Expected `.env` inside `day1-llm-foundations`:

```env
OPENAI_API_KEY=sk-your-key-here
```

Optional later:

```env
DEEPSEEK_API_KEY=your-key
ANTHROPIC_API_KEY=your-key
GOOGLE_API_KEY=your-key
```

In [3]:
repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

env_path = repo_root / ".env"

print("Repository root:", repo_root)
print("Loading .env from:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)

openai_key = os.getenv("OPENAI_API_KEY")

if not openai_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError(f"OPENAI_API_KEY not found in {env_path}")

print("OPENAI_API_KEY loaded successfully.")

Repository root: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai
Loading .env from: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.env
Env exists: True
OPENAI_API_KEY loaded successfully.


In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("OpenAI client initialized successfully.")

OpenAI client initialized successfully.


# 1️⃣ What is an API?

An API is a bridge between your application and another service.

```text
Your Python Code
      ↓
LLM API
      ↓
OpenAI / Claude / Gemini / DeepSeek
      ↓
Model Response
```

In simple terms:

> Your code sends a prompt. The model sends back a response.

# 2️⃣ Where Does the Model Run?

| Option | Where it runs | Examples | Best For |
|---|---|---|---|
| ☁️ Cloud | Provider servers | OpenAI, Claude, Gemini, DeepSeek API | Best quality, easiest start |
| 💻 Local | Your laptop/server | Ollama, LM Studio, llama.cpp | Privacy, offline use, zero API cost |
| 🏢 Hybrid | Mix of cloud and local | Enterprise architecture | Cost, compliance, flexibility |

Key idea:

> Choosing a model is not only about intelligence. It is also about **cost, privacy, speed and control**.

 Your First LLM API Call

Until now, we have only configured the client.

Now we will send our first request to a hosted Large Language Model.

Every Chat Completion request contains:

- Model
- Messages
- Parameters (temperature, max_tokens, etc.)

The model returns a structured response containing one or more generated messages.

# 3️⃣ Your First OpenAI API Call

In [4]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain APIs in one simple paragraph for engineering faculty."}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

APIs, or Application Programming Interfaces, are sets of rules and protocols that allow different software applications to communicate with each other. They define the methods and data formats that programs can use to request and exchange information, enabling developers to integrate functionalities from one application into another without needing to understand the underlying code. This modular approach promotes efficiency, as engineers can leverage existing services and tools to build complex systems more easily, fostering innovation and collaboration across different platforms and technologies.


# 4️⃣ Message Roles: System, User and Assistant

| Role | Meaning |
|---|---|
| `system` | Instructions, persona and rules |
| `user` | Human question or task |
| `assistant` | Previous model response / conversation history |

Important:

> The assistant message is not a separate prompt. It represents what the AI said earlier.

In [5]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful AI tutor for engineering students. Use simple examples."
    },
    {
        "role": "user",
        "content": "Explain Newton's Second Law in simple terms."
    }
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3
)

print(response.choices[0].message.content)

Newton's Second Law of Motion states that the force acting on an object is equal to the mass of that object multiplied by its acceleration. This can be expressed with the formula:

\[ F = m \times a \]

Where:
- \( F \) is the force (measured in Newtons, N),
- \( m \) is the mass of the object (measured in kilograms, kg),
- \( a \) is the acceleration (measured in meters per second squared, m/s²).

### Simple Example:

Imagine you have a toy car. If you want to push it to make it go faster (accelerate), you need to apply a force. 

1. **If the car is light (small mass)**: You can push it gently, and it will accelerate quickly. For example, if the car weighs 1 kg and you push it with a force of 2 N, it will accelerate at 2 m/s² (because \( 2 N = 1 kg \times 2 m/s² \)).

2. **If the car is heavy (large mass)**: You would need to push much harder to get it to accelerate at the same rate. If the car weighs 4 kg and you push it with the same force of 2 N, it will only accelerate at 0.5 m/s²

## 🧪 Mini Demo: Why Assistant History Matters

In [6]:
# Case 1: No conversation history
messages_without_history = [
    {"role": "user", "content": "Give another example."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_without_history,
    temperature=0.3
)

print(response.choices[0].message.content)

Sure! Could you please specify what type of example you are looking for? It could be related to a specific topic, concept, or situation. Let me know so I can provide the most relevant example for you!


In [7]:
# Case 2: With conversation history
messages_with_history = [
    {"role": "user", "content": "Explain Newton's Second Law in simple terms."},
    {"role": "assistant", "content": "Newton's Second Law says that force equals mass times acceleration."},
    {"role": "user", "content": "Give another example."}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_with_history,
    temperature=0.3
)

print(response.choices[0].message.content)

Sure! Imagine you have a shopping cart. If the cart is empty (light), you can push it easily, and it will move quickly. But if the cart is full of heavy groceries (heavy), you have to push much harder to get it moving. 

In this example:

- The **force** you apply is how hard you push the cart.
- The **mass** is how heavy the cart is (empty vs. full).
- The **acceleration** is how fast the cart speeds up when you push it.

So, if you want to make the cart go faster (more acceleration), you either need to push harder (more force) or lighten the load (less mass). This illustrates Newton's Second Law: the more mass an object has, the more force you need to accelerate it.


## 💬 Discussion

Ask participants:

1. Why did the first answer lack context?
2. Why did the second answer understand "another example"?
3. What happens if a conversation becomes very long?
4. How does this connect to context window?

# 5️⃣ Reusable Helper Function

In [9]:
def ask_openai(
    prompt,
    model="gpt-4o-mini",
    temperature=0.3,
    system_prompt="You are a helpful AI teaching assistant.",
    max_tokens=500
):
    """
    Sends a prompt to an OpenAI chat model and returns the response text.
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [10]:
print(ask_openai("Explain the difference between AI, ML and Deep Learning in 5 bullet points."))

Sure! Here are five bullet points that explain the differences between Artificial Intelligence (AI), Machine Learning (ML), and Deep Learning (DL):

1. **Definition**:
   - **AI**: Refers to the broader concept of machines being able to perform tasks that typically require human intelligence, such as reasoning, problem-solving, and understanding natural language.
   - **ML**: A subset of AI that focuses on the development of algorithms that allow computers to learn from and make predictions or decisions based on data.
   - **DL**: A subset of ML that utilizes neural networks with many layers (deep networks) to analyze various forms of data, particularly unstructured data like images, audio, and text.

2. **Complexity**:
   - **AI**: Encompasses a wide range of techniques and approaches, including rule-based systems, expert systems, and more.
   - **ML**: Involves statistical techniques and algorithms that improve performance on a task as more data is provided.
   - **DL**: Involves com

# 6️⃣ Same Prompt, Different Temperatures

In [11]:
prompt = "Create one catchy title for a 5-day FDP on Agentic AI for AI and ML faculty."

for temp in [0.0, 0.3, 0.7, 1.2]:
    print("=" * 80)
    print("Temperature:", temp)
    print("=" * 80)
    print(ask_openai(prompt, temperature=temp, max_tokens=100))
    print()

Temperature: 0.0
"Empowering Minds: A 5-Day FDP on Harnessing Agentic AI for Transformative Learning in AI and ML"

Temperature: 0.3
"Empowering Minds: A 5-Day FDP on Harnessing Agentic AI for Innovative Teaching in AI & ML"

Temperature: 0.7
"Empowering Minds: A 5-Day Deep Dive into Agentic AI for Future Innovators"

Temperature: 1.2
"Empowering Minds: A 5-Day Journey into Agentic AI for Faculty Innovators"



## 💬 Discussion

Which temperature would you choose for:

- Code generation?
- Creative workshop title?
- Policy summarization?
- Exam question generation?
- Student feedback?

Rule of thumb:

> Correctness needed → lower temperature  
> Creativity needed → higher temperature

# 7️⃣ Same Prompt, Different Model Choices

In [ ]:
comparison_prompt = '''
Explain Retrieval-Augmented Generation (RAG) to an AI and ML professor.
Use:
1. A one-line definition
2. A simple analogy
3. One real-world university use case
'''

models_to_test = ["gpt-4o-mini"]

# Uncomment if your account has access and you want to compare.
# models_to_test.append("gpt-4o")

for model in models_to_test:
    print("=" * 80)
    print("Model:", model)
    print("=" * 80)
    print(ask_openai(comparison_prompt, model=model, temperature=0.3, max_tokens=400))
    print()

## 🧪 AI Mythbusters #2

### Myth:
> The most expensive model is always the best model.

### Reality:
Not always.

For many teaching tasks:
- cheaper models are good enough
- faster models improve classroom experience
- local models may be better for privacy
- long-context models are useful only when needed

# 8️⃣ Streaming Output

In [12]:
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Explain why streaming output improves chatbot user experience in 5 bullet points."}
    ],
    temperature=0.3,
    stream=True
)

for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)

1. **Immediate Feedback**: Streaming output allows users to see responses in real-time as they are generated, providing immediate feedback and reducing the perceived waiting time. This creates a more engaging and interactive experience.

2. **Natural Conversation Flow**: By delivering responses incrementally, streaming mimics human conversation patterns, making interactions feel more fluid and natural. Users can process information as it comes, leading to a more intuitive dialogue.

3. **Enhanced Engagement**: The dynamic nature of streaming output keeps users engaged, as they are more likely to stay focused on the conversation when they see responses unfolding gradually rather than receiving a complete answer all at once.

4. **Reduced Cognitive Load**: Breaking down responses into smaller, manageable pieces can help users absorb information more easily, reducing cognitive overload and allowing them to better understand and respond to the chatbot.

5. **Increased Anticipation and Inte

# 9️⃣ Multi-Turn Teaching Assistant

In [ ]:
teaching_messages = [
    {
        "role": "system",
        "content": "You are a patient AI tutor for B.Tech students. Explain clearly, ask one follow-up question, and avoid unnecessary jargon."
    }
]

def chat_with_tutor(user_message):
    """
    Adds user message, calls the model, stores assistant reply, and returns it.
    """
    teaching_messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=teaching_messages,
        temperature=0.4,
        max_tokens=500
    )

    reply = response.choices[0].message.content
    teaching_messages.append({"role": "assistant", "content": reply})
    return reply

In [ ]:
print(chat_with_tutor("Explain overfitting using a student exam preparation analogy."))

In [ ]:
print(chat_with_tutor("Now give me one example from machine learning."))

In [ ]:
print(chat_with_tutor("Summarize our discussion so far in 3 bullet points."))

## 🔍 Observe

The model understood "our discussion so far" because we sent the full message history.

Trade-off:

> Longer conversation = more tokens = more cost = more context usage.

Local Open-Source Models with Ollama

So far, every example has used a **hosted LLM (OpenAI)**.

Now let's see how we can run an **open-source model locally** on our own machine.

Hosted vs Local

| Hosted Models | Local Models |
|---------------|--------------|
| Runs in the cloud | Runs on your Mac |
| Requires Internet | Works offline after download |
| Pay per API call | No API cost |
| Provider manages infrastructure | You manage the hardware |
| Data leaves your machine | Data can remain on your machine |

For this demo we are using:

```bash
llama3.2:3b
```

This is **one** of the many open-source models supported by Ollama.


# 🔟 Local Models with Ollama

Ollama lets you run open-source models locally.



In [18]:
# This cell is optional.
# It works only if Ollama is installed and running locally.

RUN_OLLAMA = True

print("Using Local Model: llama3.2:3b")

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

response = ollama_client.chat.completions.create(
    model="llama3.2:3b",
    messages=[
        {
            "role": "system",
            "content": "You are an AI tutor."
        },
        {
            "role": "user",
            "content": "Explain Agentic AI in three simple sentences."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

Using Local Model: llama3.2:3b
Here's a brief explanation of Agentic AI:

Agentic AI refers to artificial intelligence systems that have the ability to make decisions and act autonomously, without being explicitly programmed for a specific task or goal. These systems learn from their environment and adapt to new situations, allowing them to exhibit complex behaviors and decision-making processes similar to those found in humans. By enabling autonomous action, Agentic AI has the potential to revolutionize various fields such as robotics, healthcare, and finance by creating more flexible and responsive systems.


Did you notice?

We did **not** change our application.

Only these changed:

- Model name
- Endpoint

Everything else remained exactly the same.

This is why modern AI applications should be designed to be **provider independent**.

Tomorrow we will see that frameworks like LangChain, CrewAI and OpenAI SDK all follow the same philosophy.

In [16]:
prompt = """
Explain Retrieval Augmented Generation
in one paragraph suitable for faculty.
"""

print("=" * 80)
print("GPT-4o-mini")
print("=" * 80)

gpt = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":prompt}]
)

print(gpt.choices[0].message.content)

print("\n")

print("=" * 80)
print("Llama 3.2 (Local)")
print("=" * 80)

llama = ollama_client.chat.completions.create(
    model="llama3.2:3b",
    messages=[{"role":"user","content":prompt}]
)

print(llama.choices[0].message.content)

GPT-4o-mini
Retrieval Augmented Generation (RAG) is an advanced natural language processing technique that enhances the capabilities of generative models by integrating retrieval mechanisms into the text generation process. It involves a two-step approach: first, relevant documents or information are retrieved from a large external knowledge base or corpus in response to a user's query; next, a generative model, such as a transformer, synthesizes a coherent and contextually accurate response based on both the retrieved information and the original prompt. This hybrid strategy not only improves the relevance and factual accuracy of the generated content but also enables the model to leverage up-to-date information that may not be present in its training data, making it particularly effective for tasks requiring detailed knowledge or specific data points.


Llama 3.2 (Local)
Retrieval-Augmented Generation (RA-G) is a novel framework for pre-training and downstream generation tasks, which

Other Models Available in Ollama

Today we used:

- llama3.2:3b

Some other popular models are:

| Model | Best For | Pros | Cons |
|--------|----------|------|------|
| llama3.2:1b | Small laptops | Very fast | Lower quality |
| llama3.2:3b | General purpose | Best balance of speed and quality | Limited complex reasoning |
| qwen2.5:3b | Coding & multilingual | Excellent instruction following | Slightly larger |
| qwen2.5:7b | Better reasoning | Better coding and reasoning | More RAM required |
| gemma3:4b | Google model | Strong text generation | Larger download |
| deepseek-r1 | Reasoning | Excellent reasoning | Slow, requires more memory |

For this FDP we selected **llama3.2:3b** because it provides the best balance between:

- Download size
- Speed
- Response quality
- Memory usage

# 1️⃣1️⃣ OpenAI-Compatible Provider Pattern

In [ ]:
def create_openai_compatible_client(provider):
    """
    Creates clients for OpenAI-compatible providers.
    This function is for teaching purposes.
    """
    provider = provider.lower()

    if provider == "openai":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    elif provider == "ollama":
        return OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )

    elif provider == "deepseek":
        deepseek_key = os.getenv("DEEPSEEK_API_KEY")
        if not deepseek_key:
            raise ValueError("DEEPSEEK_API_KEY not found in .env")
        return OpenAI(
            base_url="https://api.deepseek.com",
            api_key=deepseek_key
        )

    else:
        raise ValueError(f"Unknown provider: {provider}")

## Note on Claude and Gemini

Claude and Gemini have their own official SDKs.

Some gateways offer OpenAI-compatible access, but for teaching fundamentals, it is better to first learn:

- OpenAI style API
- Native SDKs where required
- Local model access through Ollama

# 1️⃣2️⃣ Cost Awareness

In [5]:
cost_data = [
    {"Scenario": "Single student explanation", "Approx Input Tokens": 300, "Approx Output Tokens": 300, "Risk": "Low"},
    {"Scenario": "100 students asking 5 questions each", "Approx Input Tokens": 300 * 500, "Approx Output Tokens": 300 * 500, "Risk": "Medium"},
    {"Scenario": "Agent running tools in a loop", "Approx Input Tokens": "Can grow quickly", "Approx Output Tokens": "Can grow quickly", "Risk": "High"},
    {"Scenario": "RAG chatbot with large documents", "Approx Input Tokens": "High because context is included", "Approx Output Tokens": "Medium", "Risk": "Medium to High"},
]

pd.DataFrame(cost_data)

,Scenario,Approx Input Tokens,Approx Output Tokens,Risk
0,Single student explanation,300,300,Low
1,100 students asking 5 questions each,150000,150000,Medium
2,Agent running tools in a loop,Can grow quickly,Can grow quickly,High
3,RAG chatbot with large documents,High because context is included,Medium,Medium to High


# 1️⃣3️⃣ Model Selection Framework

In [4]:
model_selection = [
    {"Use Case": "Quick teaching explanation", "Recommended Option": "GPT-4o-mini / Gemini Flash", "Why": "Low cost, fast, good quality"},
    {"Use Case": "Sensitive student data", "Recommended Option": "Local model using Ollama", "Why": "Data stays inside institution"},
    {"Use Case": "Long research paper analysis", "Recommended Option": "Claude / Gemini long-context model", "Why": "Large context window"},
    {"Use Case": "Complex coding support", "Recommended Option": "GPT-4o / Claude Sonnet", "Why": "Better reasoning and code quality"},
    {"Use Case": "Budget-constrained experimentation", "Recommended Option": "GPT-4o-mini / DeepSeek / Ollama", "Why": "Cost-effective"},
]

pd.DataFrame(model_selection)

,Use Case,Recommended Option,Why
0,Quick teaching explanation,GPT-4o-mini / Gemini Flash,"Low cost, fast, good quality"
1,Sensitive student data,Local model using Ollama,Data stays inside institution
2,Long research paper analysis,Claude / Gemini long-context model,Large context window
3,Complex coding support,GPT-4o / Claude Sonnet,Better reasoning and code quality
4,Budget-constrained experimentation,GPT-4o-mini / DeepSeek / Ollama,Cost-effective


## 💬 Faculty Discussion

Your college wants to build four AI systems:

1. Student FAQ chatbot  
2. Research paper summarizer  
3. Question paper generator  
4. Private HR policy assistant  

Which model option would you choose for each?

Expected discussion:
- FAQ chatbot → cheap cloud or local + RAG
- Research summarizer → long-context model
- Question paper generator → cheap frontier model, with faculty review
- HR policy assistant → local or secure enterprise setup

# 1️⃣4️⃣ AI Pulse: Why Open-Source Models Matter

Discussion topic:

> Why would companies release powerful models for free?

Possible reasons:

- Build developer ecosystem
- Compete with closed providers
- Drive hardware/cloud adoption
- Encourage research adoption
- Create standards and mindshare

Key message:

> Open-source models make AI education more accessible.

# 1️⃣5️⃣ Lab Assignment

## Build Your Multi-Provider AI Assistant

### Task 1
Call `gpt-4o-mini` with a question from your subject area.

### Task 2
Change the system prompt so the model behaves like:
- a strict examiner
- a friendly tutor
- a research guide

Compare outputs.

### Task 3
Build a 3-turn conversation using the `assistant` role as conversation history.

### Task 4
Try streaming output.

### Task 5
Optional: run the same prompt using Ollama.

### Bonus
Create a simple model selection table for your department's AI use cases.

# ✅ Key Takeaways

1. Most AI apps talk to models through APIs.
2. Cloud models are easy and powerful, but cost and privacy matter.
3. Local models are useful for privacy, offline labs and low-cost experimentation.
4. The OpenAI-style message format is widely used.
5. `system` defines behavior, `user` gives the task, and `assistant` stores previous model responses.
6. Streaming improves user experience.
7. Model choice depends on task, cost, quality, privacy and context window.
8. The most expensive model is not always the best model.